# TPN cement emission forecasting: run in Google Colab

Run the cells **from top to bottom** (Shift+Enter). The code is cloned from GitHub; nothing needs to be uploaded:

* **Step 2** — clone the code from GitHub
* **Step 4** — downloads the dataset from Zenodo automatically (or upload the five CSV files)

A CPU runtime is enough (*Runtime → Change runtime type → CPU*). Results are saved to your Google
Drive (Step 1) so they survive if Colab disconnects.

## Step 1 — Connect Google Drive (keeps results safe if the session disconnects)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/tpn_cement'      # results are copied here
os.makedirs(DRIVE_DIR, exist_ok=True)
print('Results will be saved in:', DRIVE_DIR)

## Step 2 — Get the code from GitHub

In [ ]:
import os, shutil
REPO_URL = 'https://github.com/<your-username>/cement-co2-forecasting-nbeats.git'   # <-- edit
CODE_DIR = '/content/cement-co2-forecasting-nbeats'
shutil.rmtree(CODE_DIR, ignore_errors=True)
!git clone -q {REPO_URL} {CODE_DIR}
os.chdir(CODE_DIR)
print('Code ready in', os.getcwd())
print(sorted(os.listdir('.')))

## Step 3 — Install dependencies
Colab already has PyTorch; this only adds anything missing.

In [ ]:
!pip -q install -r requirements.txt
import torch, sklearn, pandas, numpy
print('torch', torch.__version__, '| sklearn', sklearn.__version__, '| pandas', pandas.__version__)

## Step 4 — Get the dataset (GCP-CEM, Andrew)

**Use the full national series, not `4. combined_cement_data.xlsx`.** That Excel file holds only the
official-reporting data used to overwrite estimates, so it covers few countries and stops early.
The model needs five CSV files from the same Zenodo release:
`0. GCP-CEM.csv`, `1. annual_cement_production.csv`, `2. annual_clinker_production.csv`, and the two
`..._source.csv` files, which record where each production value came from (used for provenance).

**Option A (default):** download them directly from Zenodo (Colab has internet access).
**Option B:** set `USE_UPLOAD = True` and upload the five CSV files from your disk.

In [ ]:
import os, shutil, subprocess
ZENODO_RECORD = '20397304'      # GCP-CEM release 260526 (May 2026); change to the release you cite
USE_UPLOAD = False              # True = upload the three CSV files yourself instead

DATA_PATH = '/content/data/gcp_cem'
os.makedirs(DATA_PATH, exist_ok=True)
FILES = ['0. GCP-CEM.csv', '1. annual_cement_production.csv', '2. annual_clinker_production.csv',
         '1. annual_cement_production_source.csv', '2. annual_clinker_production_source.csv']   # source files give provenance

if USE_UPLOAD:
    from google.colab import files
    uploaded = files.upload()                               # <-- UPLOAD HOLDER: the five CSV files
    for f in uploaded:
        shutil.move(f, os.path.join(DATA_PATH, f))
else:
    for f in FILES:
        url = f'https://zenodo.org/records/{ZENODO_RECORD}/files/{f.replace(" ", "%20")}?download=1'
        subprocess.run(['wget', '-q', '-O', os.path.join(DATA_PATH, f), url], check=True)

SHEET = 'X-Y'                   # only used for Excel input; ignored for the CSV folder
print(sorted(os.listdir(DATA_PATH)))

## Step 5 — Load and check the data
Prints the number of countries, the unit conversion applied to production data, the implied clinker
ratio and emission factor (should be about 0.6–0.9 and about 0.52), and the provenance shares.

In [ ]:
import pandas as pd
from tpn_cement.config import Config
from tpn_cement.data import load_any, assign_provenance, build_sequences, describe
cfg = Config()
panel = assign_provenance(load_any(DATA_PATH, cfg, SHEET), cfg)
both = panel.dropna(subset=['production', 'clinker'])
print('median clinker/cement ratio :', round(float((both.clinker / both.production).median()), 3))
print('median emissions/clinker    :', round(float((panel.emissions / panel.clinker).median()), 3))
S = build_sequences(panel, cfg)
print('\nReady:', describe(S, panel, cfg))

## Step 5b — Data coverage (important)
Shows, for every country, the first and last year of emissions, gaps, and how many forecast windows
it gives. A country needs 11 consecutive years to give one window. The last lines show the cut-off
years the experiment will use; they are chosen from coverage alone, before any model is trained.
With the full GCP-CEM series most countries should run to 2023.

**Also check the 'Source labels' table**: each production/clinker source label is marked *estimate*
or *reported*. Copy that table into the Supplementary Material, and send it to me if any label looks misclassified.

In [ ]:
!python -m tpn_cement diagnose --data {DATA_PATH} --sheet "{SHEET}"

## Step 6 — Self-test (about 3–5 minutes)
Checks the leakage guarantees and runs the whole pipeline on synthetic data. All lines should say **PASS**.

In [ ]:
!python -m tpn_cement selftest --out /content/selftest_out 2>&1 | grep -E "PASS|FAIL|Error|complete"

## Step 7 — Quick trial on your data (about 5–15 minutes, optional)
Two seeds, no ablation, no cold-start. Use it to see that everything works before the full run.
**These numbers are not for the paper.**

In [ ]:
!python -m tpn_cement experiment --data {DATA_PATH} --sheet "{SHEET}" --out /content/quick_run --seeds 0 1 --no-ablation --no-coldstart
import pandas as pd, json
R = json.load(open('/content/quick_run/results.json'))
print(pd.DataFrame({m: {'MASE': v['all']['MASE'], 'sMAPE %': v['all']['sMAPE']} for m, v in R['main'].items()}).T.round(3))

## Step 8 — Full experiment for the paper (roughly 1–3 hours on Colab CPU)
5 seeds, 3 folds, all baselines, ablation and cold-start. Output goes straight to Google Drive, so a
disconnect does not lose finished results. Keep the browser tab open.

*Optional:* if you have a file `meta.csv` with columns `ISO, region, income_group`, upload it and add
`--country-meta meta.csv` to the command (adds geographic/economic edges to the static-graph baseline).

In [ ]:
RESULTS_DIR = f'{DRIVE_DIR}/manuscript_results'
CUTOFFS = []          # leave empty to use the cut-offs suggested in Step 5b, e.g. [2008, 2011]
CUTOFF_ARG = ('--cutoffs ' + ' '.join(map(str, CUTOFFS))) if CUTOFFS else ''
!python -m tpn_cement experiment --data {DATA_PATH} --sheet "{SHEET}" --out "{RESULTS_DIR}" {CUTOFF_ARG} 2>&1 | tee "{DRIVE_DIR}/experiment_log.txt"

## Step 9 — Look at the main results

In [ ]:
import json, pandas as pd
from IPython.display import Image, display
R = json.load(open(f'{RESULTS_DIR}/results.json'))
print('Data:', R['data'])
rows = {m: {'MASE': v['all']['MASE'], 'sMAPE %': v['all']['sMAPE'], 'MAE kt': v['all']['MAE_kt'],
            'MASE high': v['high']['MASE'], 'MASE low': v['low']['MASE']} for m, v in R['main'].items()}
display(pd.DataFrame(rows).T.sort_values('MASE').round(3))
print('\nN-BEATS framework minus comparator (negative favours N-BEATS), country-cluster bootstrap:')
display(pd.DataFrame(R['significance'])[['model', 'diff', 'lo', 'hi', 'p', 'p_holm']].round(3))
print('\nEvery model vs persistence (negative favours the model):')
display(pd.DataFrame(R['vs_persistence'])[['model', 'diff', 'lo', 'hi', 'p_holm']].round(3))
print('\nModel with the lowest VALIDATION error per fold (rank of N-BEATS):'); display(pd.DataFrame(R['val_model_choice']).round(3))
print('\nError by emission scale and by provenance:'); display(pd.DataFrame(R['heterogeneity']).round(3))
print('\nCold start (countries withheld from training):'); display(pd.DataFrame({k: {'MASE': v['MASE'], 'sMAPE': v['sMAPE']} for k, v in R['coldstart'].items()}).T.round(3))
cal = pd.DataFrame(R['calibration']); print('\nInterval coverage:'); display(cal[cal.nominal.round(2).isin([0.8, 0.9])][['model','method','nominal','all_coverage','high_coverage','low_coverage','lastfold_coverage','all_width_pct']].round(3))
print('\nTPN variant selected on validation data per fold:'); display(pd.DataFrame(R['val_selected']))
print('\nAblation:'); display(pd.DataFrame(R['ablation']).round(3))
for f in ['fig_forest', 'fig_heterogeneity', 'fig_calibration', 'fig_examples', 'fig_perfold']:
    display(Image(f'{RESULTS_DIR}/figures/{f}.png', width=550))

## Step 9b — Figure 1 of the manuscript (data overview)
Needs only Steps 1–3 (Drive, code, install); it downloads the data itself if needed.

In [ ]:
# Self-contained: works even if Steps 4-9 were skipped or the runtime restarted.
import os, subprocess
from IPython.display import Image, display

DATA_PATH = globals().get('DATA_PATH', '/content/data/gcp_cem')
SHEET = globals().get('SHEET', 'X-Y')
DRIVE_DIR = globals().get('DRIVE_DIR', '/content/drive/MyDrive/tpn_cement')
RESULTS_DIR = globals().get('RESULTS_DIR', f'{DRIVE_DIR}/manuscript_results')
os.makedirs(RESULTS_DIR, exist_ok=True)

# download the GCP-CEM files if they are not there yet
FILES = ['0. GCP-CEM.csv', '1. annual_cement_production.csv', '2. annual_clinker_production.csv',
         '1. annual_cement_production_source.csv', '2. annual_clinker_production_source.csv']
os.makedirs(DATA_PATH, exist_ok=True)
for f in FILES:
    dest = os.path.join(DATA_PATH, f)
    if not os.path.exists(dest) or os.path.getsize(dest) == 0:
        url = f'https://zenodo.org/records/20397304/files/{f.replace(" ", "%20")}?download=1'
        subprocess.run(['wget', '-q', '-O', dest, url], check=True)

res = subprocess.run(['python', '-m', 'tpn_cement', 'overview', '--data', DATA_PATH,
                      '--sheet', SHEET, '--out', RESULTS_DIR], capture_output=True, text=True)
print(res.stdout[-2000:], res.stderr[-2000:])
fig = f'{RESULTS_DIR}/figures/fig_overview.png'
if os.path.exists(fig):
    display(Image(fig, width=750))
    print('Saved:', fig.replace('.png', '.pdf'))

## Step 10 — Train the deployable model and make a test forecast

In [ ]:
ART_DIR = f'{DRIVE_DIR}/artefact'
# Deploy the proposed framework: N-BEATS seed ensemble with provenance-stratified conformal intervals
!python -m tpn_cement train-final --data {DATA_PATH} --sheet "{SHEET}" --out "{ART_DIR}" --backtest "{RESULTS_DIR}/results.json" --model N-BEATS

# Example: forecast for one country from its own history in the dataset
from tpn_cement.config import Config
from tpn_cement.data import load_any
from tpn_cement.deploy import Forecaster
panel = load_any(DATA_PATH, Config(), SHEET)
COUNTRY = 'MEX'                                            # <-- change to any ISO code in your data
if COUNTRY not in set(panel['iso']):
    COUNTRY = panel['iso'].iloc[0]
hist = panel[panel['iso'] == COUNTRY][['year', 'emissions', 'production', 'clinker']]
fc = Forecaster.load(f'{ART_DIR}/tpn_artifact.pt')
display(fc.predict(hist, iso=COUNTRY).round(1))
print(open(f'{ART_DIR}/model_card.md').read())

## Step 11 — Download everything

The zip contains `results_macros.tex`, `tables/`, `figures/`, `results.json`, `test_predictions.csv`
and the deployable model.

In [ ]:
import shutil
from google.colab import files
shutil.make_archive('/content/tpn_results', 'zip', DRIVE_DIR)
files.download('/content/tpn_results.zip')